# 🏢 CORP-ENV · Qwen 2.5-3B-Instruct — SFT + RLVR Training

**End-to-end reproducible notebook** for training a Qwen 2.5-3B-Instruct agent on CORP-ENV using Supervised Fine-Tuning (SFT) followed by Rejection-Sampling RL on Verifiable Rewards (RLVR).

### ⚡ Optimized for Google Colab T4 (16 GB VRAM)

This notebook is configured to run end-to-end on a **free-tier T4 GPU**:
- 4-bit QLoRA quantization to fit 7B model in ~4 GB VRAM
- **FP16** precision (T4 lacks BF16 hardware support)
- Reduced sequence lengths (4096 tokens) and RLVR samples (4 per prompt)
- Inline visualizations after every training and evaluation step

CORP-ENV is a multi-agent corporate decision environment where a Master Agent governs a **Shared Workspace Document (SWD)** across long-horizon planning episodes, coordinating frozen worker agents. Rewards measure SWD integrity, task completion, milestone adherence, reasoning density, and LLM-judge scores.

| Component | Detail |
|---|---|
| **Base model** | `Qwen/Qwen2.5-3B-Instruct` |
| **SFT script** | `training/train_sft.py` |
| **RLVR script** | `training/train_rlvr.py` |
| **Tasks** | E1 Launch Readiness, M1 Budget Reallocation, H1 Acquisition Defence |
| **Runtime** | ✅ Google Colab T4 / Lightning AI H100 / Any CUDA GPU |

---

## 1️⃣ Setup & Installation

In [ ]:
import os
import torch

# ===== GPU Detection & Configuration =====
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    has_bf16 = torch.cuda.is_bf16_supported()
    print(f"🖥️  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"   BF16 support: {'✅ Yes' if has_bf16 else '❌ No (using FP16)'}")
else:
    raise RuntimeError("❌ No GPU detected! Enable GPU in Colab: Runtime → Change runtime type → T4 GPU")

# Auto-detect hardware constraints
LOW_MEMORY = gpu_mem < 20.0  # e.g., T4 (16GB), RTX 4080 (16GB) need smaller batches/sequences
USE_FP16 = not has_bf16      # e.g., T4 and V100 dont support BF16

# ===== Configuration =====
REPO_URL = "https://huggingface.co/spaces/Navigam/corp-env"  # Change to your repo
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
HF_ORG_OR_USER = "Navigam"  # Your HF username/org

# SFT hyperparameters (T4-optimized)
SFT_MAX_STEPS = 30         # Quick judge smoke; set -1 for full-epoch training
SFT_EPOCHS = 2.0
SFT_LR = 2e-4
SFT_BATCH_SIZE = 1
SFT_GRAD_ACCUM = 8
        "SFT_MAX_SEQ_LEN = 3072 if LOW_MEMORY else 8192  # Reduced for <20GB VRAM\n",

# RLVR hyperparameters (T4-optimized)
RLVR_ROUNDS = 3
RLVR_MAX_PROMPTS = 32 if LOW_MEMORY else 128   # Fewer prompts to fit in T4 time/memory
        "RLVR_N_SAMPLES = 4 if LOW_MEMORY else 8         # Fewer samples per prompt\n",
RLVR_TEMPERATURE = 0.7
        "RLVR_MAX_PROMPT_LEN = 3072 if LOW_MEMORY else 8192\n",
RLVR_MAX_COMPLETION_LEN = 512

# Eval
EVAL_EPISODES = 3
EVAL_MAX_STEPS = 30

# FP16 flag for training scripts
FP16_FLAG = "--fp16" if USE_FP16 else ""

print(f"\n📋 Config: model={BASE_MODEL}, fp16={USE_FP16}, seq_len={SFT_MAX_SEQ_LEN}")
print(f"   RLVR: rounds={RLVR_ROUNDS}, prompts={RLVR_MAX_PROMPTS}, samples={RLVR_N_SAMPLES}")

In [ ]:
# ===== Install dependencies (Colab-optimized) =====
# Unsloth requires a specific install path for Colab
import subprocess, sys

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Installing Unsloth for Colab...")
    !pip install -q --no-deps trl peft accelerate bitsandbytes triton
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q --no-deps unsloth_zoo
    !pip install -q xformers
else:
    print("🔧 Installing from pyproject.toml...")
    !pip install -q -U pip

# Clone and install CORP-ENV
!git clone {REPO_URL} corp_gym 2>/dev/null || echo 'Repo already cloned'
%cd corp_gym
!pip install -q -e ".[training,plots]"

## 2️⃣ Hugging Face Login (optional)

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 📊 Visualization Utilities

Helper functions for inline charts after every eval and training step.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.display import display, Markdown, HTML

# ---- Plotting style ----
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'font.family': 'sans-serif',
    'font.size': 11,
})

PALETTE = {
    'baseline': '#8b949e',
    'oracle': '#a371f7',
    'sft': '#3fb950',
    'rlvr': '#f0883e',
    'e1_launch_readiness': '#58a6ff',
    'm1_budget_reallocation': '#d2a8ff',
    'h1_acquisition_defence': '#7ee787',
}
TASK_SHORT = {
    'e1_launch_readiness': 'E1 Launch',
    'm1_budget_reallocation': 'M1 Budget',
    'h1_acquisition_defence': 'H1 Acquisition',
}

def load_eval_jsonl(path):
    """Load evaluation JSONL file."""
    rows = []
    p = Path(path)
    if p.is_dir():
        for f in sorted(p.rglob('*_eval.jsonl')):
            rows.extend(load_eval_jsonl(f))
        for f in sorted(p.rglob('eval.jsonl')):
            rows.extend(load_eval_jsonl(f))
        return rows
    if p.exists():
        for line in p.read_text(encoding='utf-8').strip().splitlines():
            if line.strip():
                rows.append(json.loads(line))
    return rows

def plot_eval_dashboard(rows, title="Evaluation Results"):
    """Create a 2x2 dashboard of evaluation metrics."""
    if not rows:
        print("⚠️  No evaluation data to plot.")
        return

    # Group by task
    by_task = defaultdict(list)
    for r in rows:
        by_task[r['task_id']].append(r)

    tasks = sorted(by_task.keys())
    task_labels = [TASK_SHORT.get(t, t) for t in tasks]

    # Compute metrics
    avg_reward = [np.mean([r['terminal_reward'] for r in by_task[t]]) for t in tasks]
    avg_pass = [np.mean([r['verifier_pass_rate'] for r in by_task[t]]) for t in tasks]
    success_rate = [np.mean([1 if r.get('success') else 0 for r in by_task[t]]) for t in tasks]
    avg_steps = [np.mean([r.get('steps', 0) for r in by_task[t]]) for t in tasks]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(title, fontsize=18, fontweight='bold', color='#f0f6fc', y=0.98)

    # -- Terminal Reward --
    ax = axes[0, 0]
    colors = [PALETTE.get(t, '#58a6ff') for t in tasks]
    bars = ax.bar(task_labels, avg_reward, color=colors, edgecolor='#30363d', linewidth=0.8)
    for bar, val in zip(bars, avg_reward):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#f0f6fc')
    ax.set_title('Terminal Reward', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.grid(axis='y', alpha=0.3)

    # -- Verifier Pass Rate --
    ax = axes[0, 1]
    bars = ax.bar(task_labels, avg_pass, color=colors, edgecolor='#30363d', linewidth=0.8)
    for bar, val in zip(bars, avg_pass):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#f0f6fc')
    ax.set_title('Verifier Pass Rate', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.grid(axis='y', alpha=0.3)

    # -- Success Rate --
    ax = axes[1, 0]
    bars = ax.bar(task_labels, success_rate, color=colors, edgecolor='#30363d', linewidth=0.8)
    for bar, val in zip(bars, success_rate):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#f0f6fc')
    ax.set_title('Success Rate', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.25)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.grid(axis='y', alpha=0.3)

    # -- Avg Steps --
    ax = axes[1, 1]
    bars = ax.bar(task_labels, avg_steps, color=colors, edgecolor='#30363d', linewidth=0.8)
    for bar, val in zip(bars, avg_steps):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#f0f6fc')
    ax.set_title('Average Steps per Episode', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    for ax in axes.flat:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # Print summary table
    display(Markdown("### 📋 Summary Table"))
    header = "| Task | Terminal Reward | Verifier Pass | Success Rate | Avg Steps |"
    sep    = "|------|---------------|--------------|-------------|----------|"
    lines = [header, sep]
    for i, t in enumerate(tasks):
        lines.append(f"| {TASK_SHORT.get(t, t)} | {avg_reward[i]:.3f} | {avg_pass[i]:.3f} | {success_rate[i]:.0%} | {avg_steps[i]:.1f} |")
    display(Markdown('\n'.join(lines)))


def plot_reward_traces(rows, title="Reward Traces"):
    """Plot reward curves over episode steps."""
    traces_by_task = defaultdict(list)
    for r in rows:
        trace = r.get('reward_trace', [])
        if trace:
            traces_by_task[r['task_id']].append([float(x) for x in trace])

    if not traces_by_task:
        return

    fig, ax = plt.subplots(figsize=(12, 5))
    for task_id, traces in sorted(traces_by_task.items()):
        max_len = max(len(t) for t in traces)
        means = []
        for idx in range(max_len):
            vals = [t[idx] for t in traces if idx < len(t)]
            means.append(np.mean(vals))
        xs = range(1, max_len + 1)
        color = PALETTE.get(task_id, '#58a6ff')
        ax.plot(xs, means, marker='o', linewidth=2.2, markersize=4,
                label=TASK_SHORT.get(task_id, task_id), color=color)
        if len(traces) > 1:
            mins = [min(t[i] for t in traces if i < len(t)) for i in range(max_len)]
            maxs = [max(t[i] for t in traces if i < len(t)) for i in range(max_len)]
            ax.fill_between(xs, mins, maxs, alpha=0.15, color=color)

    ax.set_title(title, fontsize=15, fontweight='bold')
    ax.set_xlabel('Environment Step')
    ax.set_ylabel('Step Reward')
    ax.axhline(0, color='#484f58', linewidth=0.8, alpha=0.5)
    ax.legend(frameon=False, fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    plt.show()


def plot_stage_comparison(all_evals, metric='terminal_reward', title='Model Stage Comparison'):
    """Compare multiple evaluation stages side-by-side."""
    if not all_evals:
        return

    stages = list(all_evals.keys())
    all_tasks = sorted({r['task_id'] for rows in all_evals.values() for r in rows})
    task_labels = [TASK_SHORT.get(t, t) for t in all_tasks]

    x = np.arange(len(all_tasks))
    width = 0.8 / max(len(stages), 1)

    fig, ax = plt.subplots(figsize=(max(10, len(all_tasks) * 3), 6))
    for idx, stage in enumerate(stages):
        rows = all_evals[stage]
        by_task = defaultdict(list)
        for r in rows:
            by_task[r['task_id']].append(float(r.get(metric, 0)))
        vals = [np.mean(by_task.get(t, [0])) for t in all_tasks]
        offsets = x - 0.4 + width/2 + idx * width
        color = PALETTE.get(stage, f'C{idx}')
        bars = ax.bar(offsets, vals, width, label=stage.upper(), color=color,
                      edgecolor='#30363d', linewidth=0.8)
        for bar, val in zip(bars, vals):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                        f'{val:.2f}', ha='center', va='bottom', fontsize=9,
                        fontweight='bold', color='#f0f6fc')

    ax.set_title(title, fontsize=16, fontweight='bold', color='#f0f6fc')
    ax.set_xticks(x)
    ax.set_xticklabels(task_labels)
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_ylim(0, 1.15)
    ax.legend(frameon=False, fontsize=10, loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=len(stages))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    plt.show()


def plot_rlvr_stats(stats_file):
    """Plot RLVR training stats per round."""
    p = Path(stats_file)
    if not p.exists():
        print(f"⚠️  Stats file not found: {stats_file}")
        return

    stats = [json.loads(line) for line in p.read_text().strip().splitlines() if line.strip()]
    if not stats:
        return

    rounds = [s['round'] for s in stats]
    keep_rates = [s['keep_rate'] for s in stats]
    mean_best = [s['mean_best_reward'] for s in stats]
    mean_any = [s['mean_sample_reward'] for s in stats]
    kept_counts = [int(s['prompts_kept']) for s in stats]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('RLVR Training Progress', fontsize=16, fontweight='bold', color='#f0f6fc', y=1.02)

    # Keep rate
    ax = axes[0]
    ax.plot(rounds, keep_rates, marker='o', linewidth=2.5, color='#3fb950', markersize=8)
    ax.fill_between(rounds, keep_rates, alpha=0.15, color='#3fb950')
    ax.set_title('Keep Rate per Round', fontweight='bold')
    ax.set_xlabel('Round')
    ax.set_ylabel('Keep Rate')
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.grid(alpha=0.3)

    # Reward progression
    ax = axes[1]
    ax.plot(rounds, mean_best, marker='s', linewidth=2.5, color='#f0883e', markersize=8, label='Best')
    ax.plot(rounds, mean_any, marker='D', linewidth=2.5, color='#58a6ff', markersize=7, label='Any sample')
    ax.set_title('Mean Reward per Round', fontweight='bold')
    ax.set_xlabel('Round')
    ax.set_ylabel('Reward')
    ax.legend(frameon=False)
    ax.grid(alpha=0.3)

    # Prompts kept
    ax = axes[2]
    ax.bar(rounds, kept_counts, color='#a371f7', edgecolor='#30363d', linewidth=0.8)
    for r, c in zip(rounds, kept_counts):
        ax.text(r, c + 0.5, str(c), ha='center', fontweight='bold', fontsize=11, color='#f0f6fc')
    ax.set_title('Prompts Kept (Winners)', fontweight='bold')
    ax.set_xlabel('Round')
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.3)

    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    fig.tight_layout()
    plt.show()

    # Print per-round summary
    display(Markdown("### 📋 RLVR Round Summary"))
    header = "| Round | Keep Rate | Mean Best Reward | Mean Any Reward | Prompts Kept | Time (s) |"
    sep    = "|-------|-----------|-----------------|----------------|-------------|----------|"
    lines = [header, sep]
    for s in stats:
        lines.append(f"| {s['round']} | {s['keep_rate']:.1%} | {s['mean_best_reward']:.3f} | {s['mean_sample_reward']:.3f} | {int(s['prompts_kept'])} | {s['seconds']:.0f} |")
    display(Markdown('\n'.join(lines)))


def gpu_status():
    """Print current GPU memory usage."""
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        cached = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_mem / 1e9
        pct = alloc / total * 100
        bar_len, filled = 20, int(pct / 5)
        bar = '█' * filled + '░' * (bar_len - filled)
        print(f"🖥️  GPU Memory: [{bar}] {alloc:.1f}/{total:.1f} GB ({pct:.0f}%) | Cached: {cached:.1f} GB")


# Collect all eval results for final comparison
ALL_EVALS = {}
print("✅ Visualization utilities loaded.")

## 3️⃣ Environment Validation

Quick sanity check: tests pass and `openenv validate` succeeds.

In [ ]:
!python -m pytest tests/ -v --tb=short 2>/dev/null || echo 'No tests found or tests skipped'
!openenv validate

## 4️⃣ Data Preparation

Verify raw trajectory examples through the real `CorpEnvironment`, filter bad trajectories, and prepare the SFT dataset.

In [ ]:
# Import and verify E1/M1 examples
!python scripts/verify_examples.py \
  --input data/raw/e1_m1_examples.jsonl \
  --clean data/processed/e1_m1_clean.jsonl \
  --rejected data/processed/e1_m1_rejected.jsonl \
  --summary results/e1_m1_verification_summary.json \
  --all-records results/e1_m1_verification_all.jsonl

# Generate and verify H1 seed data
!python scripts/generate_sft_data.py \
  --tasks h1_acquisition_defence --per-task 24 \
  --output data/raw/h1_seed.jsonl

!python scripts/verify_examples.py \
  --input data/raw/h1_seed.jsonl \
  --clean data/processed/h1_seed_clean.jsonl \
  --rejected data/processed/h1_seed_rejected.jsonl

# Merge into final SFT dataset
!python scripts/prepare_sft_data.py

In [ ]:
# Check data stats & visualize
sft_path = Path("data/sft/e1_m1_h1_examples.jsonl")
if sft_path.exists():
    lines = [json.loads(l) for l in sft_path.read_text().strip().splitlines() if l.strip()]
    print(f"\n✅ SFT dataset: {len(lines)} examples")
    turn_counts = [len(ex['messages']) for ex in lines]
    print(f"   Avg turns per example: {sum(turn_counts)/len(turn_counts):.1f}")
    print(f"   Min/Max turns: {min(turn_counts)} / {max(turn_counts)}")

    # Visualize data distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('SFT Dataset Overview', fontsize=14, fontweight='bold', color='#f0f6fc')

    # Turns histogram
    axes[0].hist(turn_counts, bins=range(min(turn_counts), max(turn_counts)+2),
                 color='#58a6ff', edgecolor='#30363d', alpha=0.85)
    axes[0].set_title('Message Turns per Example', fontweight='bold')
    axes[0].set_xlabel('Number of Turns')
    axes[0].set_ylabel('Count')
    axes[0].grid(axis='y', alpha=0.3)

    # Role distribution
    role_counts = defaultdict(int)
    for ex in lines:
        for msg in ex['messages']:
            role_counts[msg['role']] += 1
    roles = list(role_counts.keys())
    counts = list(role_counts.values())
    role_colors = ['#a371f7', '#3fb950', '#58a6ff', '#f0883e'][:len(roles)]
    axes[1].barh(roles, counts, color=role_colors, edgecolor='#30363d')
    axes[1].set_title('Messages by Role', fontweight='bold')
    axes[1].set_xlabel('Count')
    axes[1].grid(axis='x', alpha=0.3)

    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    fig.tight_layout()
    plt.show()
else:
    print("❌ SFT dataset not found. Check data preparation above.")

## 5️⃣ Baseline Evaluation

Run scripted (weak) and oracle policies to establish comparison baselines.

In [ ]:
!python eval.py --policy scripted_weak --label baseline --episodes {EVAL_EPISODES}
!python eval.py --policy oracle --label oracle --episodes {EVAL_EPISODES}

In [ ]:
# 📊 Visualize baseline results
display(Markdown("## 📊 Baseline Results"))

baseline_rows = load_eval_jsonl("results/runs")
baseline_only = [r for r in baseline_rows if r.get('model_stage') == 'baseline']
oracle_only = [r for r in baseline_rows if r.get('model_stage') == 'oracle']

if baseline_only:
    display(Markdown("### 🔹 Scripted Weak Baseline"))
    plot_eval_dashboard(baseline_only, title="Scripted Weak Baseline")
    plot_reward_traces(baseline_only, title="Baseline Reward Traces")
    ALL_EVALS['baseline'] = baseline_only

if oracle_only:
    display(Markdown("### 🔹 Oracle Policy"))
    plot_eval_dashboard(oracle_only, title="Oracle Policy")
    plot_reward_traces(oracle_only, title="Oracle Reward Traces")
    ALL_EVALS['oracle'] = oracle_only

# Side-by-side comparison if both exist
if baseline_only and oracle_only:
    plot_stage_comparison(
        {'baseline': baseline_only, 'oracle': oracle_only},
        metric='terminal_reward',
        title='Baseline vs Oracle — Terminal Reward'
    )
gpu_status()

## 6️⃣ SFT Training (Unsloth + TRL)

Fine-tune Qwen 2.5-3B-Instruct with LoRA using verified CORP-ENV trajectories.

- Uses `unsloth.FastLanguageModel` for 4-bit QLoRA
- Uses `trl.SFTTrainer` with messages-format conversational SFT
- LoRA `r=32`, targets all attention + MLP projections
- **FP16 on T4** (auto-detected), BF16 on Ampere+ GPUs

In [ ]:
gpu_status()
print(f"\n🚀 Starting SFT training ({FP16_FLAG or 'bf16'} precision)...\n")

!python training/train_sft.py \
  --model {BASE_MODEL} \
  --data data/sft/e1_m1_h1_examples.jsonl \
  --output outputs/sft_adapter \
  --max-seq-length {SFT_MAX_SEQ_LEN} \
  --max-steps {SFT_MAX_STEPS} \
  --epochs {SFT_EPOCHS} \
  --lr {SFT_LR} \
  --batch-size {SFT_BATCH_SIZE} \
  --grad-accum {SFT_GRAD_ACCUM} \
  {FP16_FLAG}

gpu_status()
print("\n✅ SFT training complete!")

In [ ]:
# 📊 Visualize SFT training logs
display(Markdown("## 📊 SFT Training Summary"))

# Check for trainer_state.json
state_file = Path("outputs/sft_adapter/trainer_state.json")
if state_file.exists():
    state = json.loads(state_file.read_text())
    log_history = state.get('log_history', [])
    if log_history:
        steps = [l['step'] for l in log_history if 'loss' in l]
        losses = [l['loss'] for l in log_history if 'loss' in l]
        lrs = [l.get('learning_rate', 0) for l in log_history if 'loss' in l]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle('SFT Training Curves', fontsize=16, fontweight='bold', color='#f0f6fc')

        # Loss curve
        axes[0].plot(steps, losses, linewidth=2.5, color='#f0883e', marker='o', markersize=5)
        axes[0].set_title('Training Loss', fontweight='bold')
        axes[0].set_xlabel('Step')
        axes[0].set_ylabel('Loss')
        axes[0].grid(alpha=0.3)

        # Learning rate schedule
        axes[1].plot(steps, lrs, linewidth=2.5, color='#3fb950', marker='s', markersize=4)
        axes[1].set_title('Learning Rate Schedule', fontweight='bold')
        axes[1].set_xlabel('Step')
        axes[1].set_ylabel('Learning Rate')
        axes[1].ticklabel_format(axis='y', style='scientific', scilimits=(-4, -4))
        axes[1].grid(alpha=0.3)

        for ax in axes:
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
        fig.tight_layout()
        plt.show()

        print(f"\n📈 Final loss: {losses[-1]:.4f} at step {steps[-1]}")
else:
    print("⚠️  No trainer_state.json found; training logs unavailable.")

# Check adapter files
adapter_dir = Path("outputs/sft_adapter")
if adapter_dir.exists():
    files = list(adapter_dir.glob("*"))
    total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
    print(f"💾 Adapter saved: {len(files)} files, {total_mb:.1f} MB total")

## 7️⃣ Evaluate SFT Adapter

In [ ]:
# Clear GPU memory before loading eval model
import gc
gc.collect()
torch.cuda.empty_cache()
gpu_status()

!python eval.py \
  --policy hf \
  --label sft \
  --model {BASE_MODEL} \
  --adapter outputs/sft_adapter \
  --episodes {EVAL_EPISODES} \
  --max-steps {EVAL_MAX_STEPS}

gpu_status()

In [ ]:
# 📊 Visualize SFT evaluation results
display(Markdown("## 📊 SFT Evaluation Results"))

sft_rows = [r for r in load_eval_jsonl("results/runs") if r.get('model_stage') == 'sft']
if sft_rows:
    plot_eval_dashboard(sft_rows, title="SFT Adapter Evaluation")
    plot_reward_traces(sft_rows, title="SFT Reward Traces")
    ALL_EVALS['sft'] = sft_rows

    # Compare baseline → SFT
    display(Markdown("### 📈 Improvement: Baseline → SFT"))
    comparison = {k: v for k, v in ALL_EVALS.items() if k in ('baseline', 'oracle', 'sft')}
    if len(comparison) > 1:
        plot_stage_comparison(comparison, metric='terminal_reward',
                             title='Baseline → SFT — Terminal Reward Comparison')
        plot_stage_comparison(comparison, metric='verifier_pass_rate',
                             title='Baseline → SFT — Verifier Pass Rate')
else:
    print("⚠️  No SFT eval results found.")

## 8️⃣ RLVR Training (Rejection-Sampling FT)

RLVR improves on SFT by:
1. Sampling N completions per prompt at non-zero temperature
2. Scoring each with the real `CorpEnvironment` verifier
3. Keeping the best completion per prompt above a reward threshold
4. SFT on that curated set
5. Repeating for multiple outer rounds

This avoids the zero-variance gradient problem seen with GRPO on CORP-ENV.

> ⚡ **T4 Note**: Using `--fp16` and reduced `--n-samples` / `--max-prompts` to fit in 16 GB VRAM.

In [ ]:
# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()
gpu_status()

print(f"\n🚀 Starting RLVR training ({RLVR_ROUNDS} rounds, {RLVR_N_SAMPLES} samples/prompt)...\n")

!python training/train_rlvr.py \
  --model {BASE_MODEL} \
  --adapter outputs/sft_adapter \
  --examples data/processed/e1_m1_clean.jsonl,data/processed/h1_seed_clean.jsonl \
  --output outputs/rlvr_adapter \
  --rounds {RLVR_ROUNDS} \
  --n-samples {RLVR_N_SAMPLES} \
  --temperature {RLVR_TEMPERATURE} \
  --max-prompts {RLVR_MAX_PROMPTS} \
  --max-prompt-length {RLVR_MAX_PROMPT_LEN} \
  --max-completion-length {RLVR_MAX_COMPLETION_LEN} \
  --strict-json \
  --use-stub-workers \
  --disable-llm-judge \
  --stats-file results/runs/rlvr_qwen2.5_3b_stats.jsonl \
  {FP16_FLAG}

gpu_status()
print("\n✅ RLVR training complete!")

In [ ]:
# 📊 Visualize RLVR training progress
display(Markdown("## 📊 RLVR Training Progress"))
plot_rlvr_stats("results/runs/rlvr_qwen2.5_3b_stats.jsonl")

# Check adapter files
rlvr_dir = Path("outputs/rlvr_adapter")
if rlvr_dir.exists():
    files = list(rlvr_dir.glob("*"))
    total_mb = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
    print(f"\n💾 RLVR adapter saved: {len(files)} files, {total_mb:.1f} MB total")

## 9️⃣ Evaluate RLVR Adapter

In [ ]:
# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()
gpu_status()

!python eval.py \
  --policy hf \
  --label rlvr \
  --model {BASE_MODEL} \
  --adapter outputs/rlvr_adapter \
  --episodes {EVAL_EPISODES} \
  --max-steps {EVAL_MAX_STEPS}

gpu_status()

In [ ]:
# 📊 Visualize RLVR evaluation results
display(Markdown("## 📊 RLVR Evaluation Results"))

rlvr_rows = [r for r in load_eval_jsonl("results/runs") if r.get('model_stage') == 'rlvr']
if rlvr_rows:
    plot_eval_dashboard(rlvr_rows, title="RLVR Adapter Evaluation")
    plot_reward_traces(rlvr_rows, title="RLVR Reward Traces")
    ALL_EVALS['rlvr'] = rlvr_rows

    # Compare SFT → RLVR
    display(Markdown("### 📈 Improvement: SFT → RLVR"))
    if 'sft' in ALL_EVALS:
        plot_stage_comparison(
            {'sft': ALL_EVALS['sft'], 'rlvr': rlvr_rows},
            metric='terminal_reward',
            title='SFT → RLVR — Terminal Reward'
        )
else:
    print("⚠️  No RLVR eval results found.")

## 📊 Final Comparison: All Model Stages

Side-by-side comparison of all evaluated model stages.

In [ ]:
display(Markdown("## 📊 Full Pipeline Comparison: Baseline → Oracle → SFT → RLVR"))

if ALL_EVALS:
    # Terminal Reward comparison
    plot_stage_comparison(ALL_EVALS, metric='terminal_reward',
                         title='Terminal Reward — All Model Stages')

    # Verifier Pass Rate comparison
    plot_stage_comparison(ALL_EVALS, metric='verifier_pass_rate',
                         title='Verifier Pass Rate — All Model Stages')

    # Build final comparison table
    display(Markdown("### 📋 Final Results Table"))
    header = "| Stage | Task | Terminal Reward | Verifier Pass | Success Rate |"
    sep    = "|-------|------|---------------|--------------|-------------|"
    lines = [header, sep]
    for stage_name, stage_rows in ALL_EVALS.items():
        by_task = defaultdict(list)
        for r in stage_rows:
            by_task[r['task_id']].append(r)
        for task_id in sorted(by_task.keys()):
            task_rows = by_task[task_id]
            avg_r = np.mean([r['terminal_reward'] for r in task_rows])
            avg_p = np.mean([r['verifier_pass_rate'] for r in task_rows])
            succ = np.mean([1 if r.get('success') else 0 for r in task_rows])
            lines.append(f"| {stage_name.upper()} | {TASK_SHORT.get(task_id, task_id)} | {avg_r:.3f} | {avg_p:.3f} | {succ:.0%} |")
    display(Markdown('\n'.join(lines)))
else:
    print("⚠️  No evaluation data collected. Run the evaluation cells above.")

In [ ]:
# Also generate plots via plot_results.py for file-based output
!python plot_results.py \
  --inputs results/runs \
  --output-dir results/model_compare_qwen25_3b

In [ ]:
from IPython.display import Image, display, Markdown

plot_dir = Path("results/model_compare_qwen25_3b")
if not plot_dir.exists():
    plot_dir = Path("results/model_compare_qwen25_fresh_no_grpo_ep5rlvr")

if plot_dir.exists():
    for png in sorted(plot_dir.glob("*.png")):
        display(Markdown(f"### {png.stem.replace('_', ' ').title()}"))
        display(Image(filename=str(png), width=800))

    # Show summary table
    summary_md = plot_dir / "comparison_summary.md"
    if summary_md.exists():
        display(Markdown(summary_md.read_text()))
else:
    print("⚠️  No plot directory found.")

## 📋 Results Summary

Expected progression for Qwen 2.5-3B-Instruct on CORP-ENV:

| Stage | E1 Terminal Reward | M1 Terminal Reward | H1 Terminal Reward | M1 Success |
|-------|-------------------|-------------------|-------------------|------------|
| Baseline (weak) | 0.590 | 0.198 | 0.257 | 0% |
| Base (pre-trained) | 0.910 | 0.765 | 0.810 | 0% |
| SFT | 0.910 | 0.943 | 0.889 | 100% |
| RLVR | 0.910 | 0.932 | 0.779 | 80% |

> **Key takeaway**: SFT dramatically improves M1 (budget reallocation) from 0% to 100% success rate. RLVR maintains strong performance while reducing reliance on fixed trajectories.

> **T4 Note**: Results may differ slightly on T4 due to FP16 precision (vs BF16) and reduced RLVR sampling. For best results, use the full hyperparameters on an A100/H100.